# MCQ Generation

Generates multiple-choice questions (1 correct answer + 3 distractors) directly from the
full lecture text of each document, using `*_unified_text.md` files under `output/`.

In [ ]:
import glob
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
UNIFIED_TEXT_GLOB = str(REPO_ROOT / "output" / "*" / "auto" / "*_unified_text.md")

WINDOW_CHARS = 3500
MIN_WINDOW_CHARS = 500  # drop a trailing tail window too small to be a real MCQ source

## Load and window each document

Each `*_unified_text.md` is the full lecture text for one document — too large for a single
MCQ-generation prompt (30KB–230KB per file). Split into fixed-size character windows on
paragraph boundaries so each prompt sees a self-contained slice of the lecture.

In [ ]:
from dataclasses import dataclass


@dataclass
class TextWindow:
    uid: str
    doc_name: str
    window_index: int
    text: str


def split_into_windows(text: str, window_chars: int = WINDOW_CHARS) -> list[str]:
    """Greedily packs paragraphs (blank-line separated) into ~window_chars windows,
    never splitting a paragraph across two windows unless the paragraph alone exceeds
    window_chars (then it becomes its own oversized window)."""
    paragraphs = [p for p in text.split("\n\n") if p.strip()]
    windows: list[str] = []
    current: list[str] = []
    current_len = 0

    for para in paragraphs:
        para_len = len(para)
        if current and current_len + para_len + 2 > window_chars:
            windows.append("\n\n".join(current))
            current, current_len = [], 0
        current.append(para)
        current_len += para_len + 2

    if current:
        windows.append("\n\n".join(current))

    return windows


def doc_name_from_path(path: str) -> str:
    return Path(path).name.replace("_v1_unified_text.md", "").replace("_unified_text.md", "")


def load_windows(pattern: str = UNIFIED_TEXT_GLOB) -> list[TextWindow]:
    paths = sorted(glob.glob(pattern))
    assert paths, f"no unified_text.md files matched {pattern!r}"

    all_windows: list[TextWindow] = []
    for path in paths:
        doc_name = doc_name_from_path(path)
        text = Path(path).read_text(encoding="utf-8")
        for i, window_text in enumerate(split_into_windows(text)):
            if len(window_text) < MIN_WINDOW_CHARS:
                continue
            all_windows.append(
                TextWindow(uid=f"{doc_name}::w{i:03d}", doc_name=doc_name, window_index=i, text=window_text)
            )

    print(f"loaded {len(paths)} doc(s), split into {len(all_windows)} window(s)")
    return all_windows


windows = load_windows()

## Generate one MCQ per window (Qwen3.5-9B)

One prompt per window: stem, correct answer, and 3 medically-plausible-but-wrong
distractors, all grounded in the window's own text. `AutoModelForMultimodalLM` +
`AutoProcessor` since Qwen3.5 is a multimodal checkpoint, run here text-only.

In [ ]:
from pydantic import BaseModel, Field, field_validator


class MCQQuestion(BaseModel):
    stem: str = Field(min_length=1)
    correct_answer: str = Field(min_length=1)
    distractors: list[str] = Field(min_length=3, max_length=3)

    @field_validator("distractors")
    @classmethod
    def distractors_differ_from_correct_answer(cls, v: list[str], info) -> list[str]:
        correct = info.data.get("correct_answer", "")
        if any(d.strip().lower() == correct.strip().lower() for d in v):
            raise ValueError("a distractor duplicates the correct answer")
        return v

In [ ]:
import gc
import json
import logging
import re

import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_PATH = "Qwen/Qwen3.5-9B"

log = logging.getLogger("mcq_generator")

_SYSTEM_PROMPT = """You are a medical education question writer. Read the given lecture \
passage and write exactly one multiple-choice question testing understanding of a fact \
explicitly stated in the passage.

The question must have:
- a clear question stem (not a fill-in-the-blank fragment)
- one correct answer, stated or directly supported by the passage
- exactly three distractors: medically plausible but factually wrong answers for this \
specific question. Prefer distractors that are other real entities/terms mentioned \
elsewhere in the passage over freely invented ones — they should match the vocabulary \
and difficulty level of the lecture content.

Do not write a question if the passage has no clear, self-contained testable fact (e.g. \
it's just a table of contents, a reference list, or a fragment with no complete idea) — \
in that case respond with {"skip": true} instead.

Respond with ONLY a JSON object of the form:
{"stem": "...", "correct_answer": "...", "distractors": ["...", "...", "..."]}
No prose, no markdown fences, no explanation — JSON only."""

_THINK_BLOCK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)
_JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)


def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def _repair_truncated_json(raw: str) -> str:
    """Best-effort fix for JSON cut off mid-generation: drops back to the last fully-formed
    value (closing an unterminated string first if needed), then closes whatever braces/
    brackets are still open — in the correct nesting order."""
    text = raw[raw.index("{"):] if "{" in raw else raw

    if text.count('"') % 2 == 1:
        text = text[: text.rindex('"')]

    last_safe = max(text.rfind("}"), text.rfind("]"), text.rfind(","))
    if last_safe == -1:
        raise ValueError("no safe truncation point found")
    text = text[:last_safe] if text[last_safe] == "," else text[: last_safe + 1]

    stack: list[str] = []
    in_string = False
    escape = False
    for ch in text:
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch in "{[":
            stack.append(ch)
        elif ch in "}]":
            stack.pop()

    closers = {"{": "}", "[": "]"}
    return text + "".join(closers[ch] for ch in reversed(stack))


class MCQGenerator:
    def __init__(self, model_path: str = MODEL_PATH):
        self.model_path = model_path
        self.device = get_device()
        self.model = None
        self.processor = None

    def load(self):
        if self.model is not None:
            return
        self.processor = AutoProcessor.from_pretrained(self.model_path)
        self.model = AutoModelForMultimodalLM.from_pretrained(
            self.model_path,
            dtype=torch.bfloat16,
        )
        self.model.to(self.device)
        self.model.eval()

    def unload(self):
        self.model = None
        self.processor = None
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()

    def generate(self, window_text: str, max_new_tokens: int = 1024, max_retries: int = 2) -> MCQQuestion | None:
        """Returns None if the model legitimately skips the window (no testable fact),
        or if every retry fails to produce a valid question."""
        assert self.model is not None, "call load() first"
        last_error: Exception | None = None
        for attempt in range(max_retries + 1):
            raw = self._generate(window_text, max_new_tokens=max_new_tokens)
            try:
                result = self._parse(raw)
            except ValueError as e:
                last_error = e
                log.warning("MCQ parse failed (attempt %d/%d): %s", attempt + 1, max_retries + 1, e)
                log.warning("raw output was: %r", raw)
                continue
            return result
        log.warning("giving up after %d attempts: %s", max_retries + 1, last_error)
        return None

    def _parse(self, raw: str) -> MCQQuestion | None:
        # Qwen3.5 is a reasoning model — strip a <think>...</think> block if the
        # thinking-disable chat template kwarg didn't take, so we don't accidentally
        # match stray braces inside the reasoning trace instead of the real answer.
        cleaned = _THINK_BLOCK_RE.sub("", raw)

        matches = _JSON_BLOCK_RE.findall(cleaned)
        if not matches:
            raise ValueError(f"no JSON object found in model output: {raw[:200]!r}")
        candidate = matches[-1]

        try:
            payload = json.loads(candidate)
        except json.JSONDecodeError:
            payload = json.loads(_repair_truncated_json(candidate))

        if payload.get("skip"):
            return None
        return MCQQuestion.model_validate(payload)

    def _generate(self, window_text: str, max_new_tokens: int) -> str:
        messages = [
            {"role": "system", "content": _SYSTEM_PROMPT},
            {"role": "user", "content": window_text},
        ]
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
        return self.processor.decode(new_tokens, skip_special_tokens=True)


generator = MCQGenerator()
generator.load()

## Generate MCQs for all windows

In [ ]:
from tqdm.auto import tqdm


@dataclass
class GeneratedMCQ:
    window: TextWindow
    question: MCQQuestion


def generate_mcqs(windows: list[TextWindow], generator: MCQGenerator) -> list[GeneratedMCQ]:
    results: list[GeneratedMCQ] = []
    skipped = 0
    for window in tqdm(windows, desc="generating MCQs"):
        question = generator.generate(window.text)
        if question is None:
            skipped += 1
            continue
        results.append(GeneratedMCQ(window=window, question=question))
    print(f"generated {len(results)}/{len(windows)} MCQ(s), {skipped} skipped/failed")
    return results

In [ ]:
# Smoke test on a small sample before running the full corpus — bump SAMPLE_SIZE to
# None to run everything.
SAMPLE_SIZE = 5
sample = windows[:SAMPLE_SIZE] if SAMPLE_SIZE else windows

mcqs = generate_mcqs(sample, generator)

In [ ]:
# Sanity check — inspect generated MCQs
for m in mcqs:
    print(m.window.uid)
    print("Q:", m.question.stem)
    print("  correct:", m.question.correct_answer)
    for d in m.question.distractors:
        print("  distractor:", d)
    print("---")

## Dump to JSON

In [ ]:
OUTPUT_DIR = Path.cwd() / "mcq_output"
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "mcqs.json"


def dump_mcqs(mcqs: list[GeneratedMCQ], path: Path) -> None:
    payload = [
        {
            "uid": m.window.uid,
            "doc_name": m.window.doc_name,
            "window_index": m.window.window_index,
            "stem": m.question.stem,
            "correct_answer": m.question.correct_answer,
            "distractors": m.question.distractors,
        }
        for m in mcqs
    ]
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"wrote {len(payload)} MCQ(s) to {path}")


dump_mcqs(mcqs, OUTPUT_PATH)